In [5]:
import requests
import pandas as pd
import time

In [3]:
r = requests.get("https://api.spacexdata.com/v4/launches/latest")
print(r.json().keys())

dict_keys(['fairings', 'links', 'static_fire_date_utc', 'static_fire_date_unix', 'net', 'window', 'rocket', 'success', 'failures', 'details', 'crew', 'ships', 'capsules', 'payloads', 'launchpad', 'flight_number', 'name', 'date_utc', 'date_unix', 'date_local', 'date_precision', 'upcoming', 'cores', 'auto_update', 'tbd', 'launch_library_id', 'id'])


In [6]:
BASE_URL      = "https://api.spacexdata.com/v4"
LAUNCHES_URL  = f"{BASE_URL}/launches"
ROCKETS_URL   = f"{BASE_URL}/rockets"
LAUNCHPADS_URL= f"{BASE_URL}/launchpads"
PAYLOADS_URL  = f"{BASE_URL}/payloads"

### GET Request

In [8]:
def get(url: str) -> dict | list:
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
            
        except requests.RequestException as e:
            if attempt == 0:
                print(f"Retrying {url} ({e})")
                time.sleep(2)
            else:
                print("Failes: {url} - {e}")
                return {} 

### Fetch data

In [9]:
launches_raw = get(LAUNCHES_URL)
rockets_raw = get(ROCKETS_URL)
pads_raw = get(LAUNCHPADS_URL)

rockets_map = {r["id"]: r["name"] for r in rockets_raw}
pads_map = {p["id"]: p["name"] for p in pads_raw}

### Flatten launch into rows

Extract the landing outcome

In [23]:
def extract_landing_outcome(cores):
    if not cores: return "Unknown"
    core = cores[0]
    ls   = core.get("landing_success")
    landing_attempt = core.get("landing_attempt")

    if ls is True: return "Success"
    if ls is False: return "Failure"
    if landing_attempt is True: return "Failure"  # attempted but no success logged = failure
    
    return "Unknown"

Get landing type

In [13]:
def extract_landpad(cores: list) -> str:

    if not cores:
        return "Unknown"
    
    return cores[0].get("landing_type") or "Unknown"

Process the launches

In [24]:
rows = []

print("processing launches...")

for launch in launches_raw:
    rocket_name = rockets_map.get(launch.get("rocket", ""), "unknown")
    if "Falcon 9" not in rocket_name:
        continue

    cores = launch.get("cores", [])
    payload_ids = launch.get("payloads", [])

    payload_mass_kg = None
    orbit = None

    if payload_ids:
        payload = get(f"{PAYLOADS_URL}/{payload_ids[0]}")
        payload_mass_kg = payload.get("mass_kg")
        orbit = payload.get("orbit")

    row = {
        # Identifiers
        "flight_number"     : launch.get("flight_number"),
        "mission_name"      : launch.get("name"),
        "date_utc"          : launch.get("date_utc"),

        # Rocket & site
        "rocket"            : rocket_name,
        "launch_site"       : pads_map.get(launch.get("launchpad", ""), "Unknown"),

        # Payload
        "payload_mass_kg"   : payload_mass_kg,
        "orbit"             : orbit,

        # Core / reuse info
        "reused"            : cores[0].get("reused") if cores else None,
        "flight_count"      : cores[0].get("flight") if cores else None,  # how many times this booster flew
        "landing_type"      : extract_landpad(cores),

        # target variable
        "landing_outcome"   : extract_landing_outcome(cores),

        # Mission success (separate from landing)
        "mission_success"   : launch.get("success"),
    }
    rows.append(row)

    # Polite delay to avoid hammering the API
    time.sleep(0.05)

print("falcon 9 launches process completed")

processing launches...
falcon 9 launches process completed


### Building the Dataframe

In [26]:
df = pd.DataFrame(rows)

df["date_utc"] = pd.to_datetime(df["date_utc"])
df["year"]     = df["date_utc"].dt.year
df["month"]    = df["date_utc"].dt.month

Encode target as binary

In [27]:
df["landing_success"] = df["landing_outcome"].map({"Success": 1, "Failure": 0})

In [28]:
df_clean = df[df["landing_outcome"] != "Unknown"].copy()

In [29]:
print(df_clean[["payload_mass_kg", "flight_count", "year"]].describe())
print(df_clean["landing_outcome"].value_counts())
print(df["landing_outcome"].value_counts())

       payload_mass_kg  flight_count         year
count       135.000000    155.000000   155.000000
mean       8751.064444      4.109677  2019.703226
std        5633.222972      3.387463     2.296735
min         330.000000      1.000000  2013.000000
25%        3050.000000      1.000000  2018.000000
50%        9600.000000      3.000000  2020.000000
75%       13260.000000      6.000000  2022.000000
max       15600.000000     14.000000  2022.000000
landing_outcome
Success    142
Failure     13
Name: count, dtype: int64
landing_outcome
Success    142
Unknown     40
Failure     13
Name: count, dtype: int64


In [30]:
df_clean.head()

,flight_number,mission_name,date_utc,rocket,launch_site,payload_mass_kg,orbit,reused,flight_count,landing_type,landing_outcome,mission_success,year,month,landing_success
5,11,CASSIOPE,2013-09-29 16:00:00+00:00,Falcon 9,VAFB SLC 4E,500.0,PO,False,1.0,Ocean,Failure,True,2013,9,0.0
8,14,CRS-3,2014-04-18 19:25:00+00:00,Falcon 9,CCSFS SLC 40,2296.0,ISS,False,1.0,Ocean,Success,True,2014,4,1.0
9,15,OG-2 Mission 1,2014-07-14 15:15:00+00:00,Falcon 9,CCSFS SLC 40,1316.0,LEO,False,1.0,Ocean,Success,True,2014,7,1.0
12,18,CRS-4,2014-09-21 05:52:00+00:00,Falcon 9,CCSFS SLC 40,2216.0,ISS,False,1.0,Ocean,Failure,True,2014,9,0.0
13,19,CRS-5,2015-01-10 09:47:00+00:00,Falcon 9,CCSFS SLC 40,2395.0,ISS,False,1.0,ASDS,Failure,True,2015,1,0.0


### Conclusion
Dataset is not good enough to classify / predict anything